# บทเรียน 08 - รูปแบบการออกแบบมัลติเอเจนต์


## การตั้งค่า


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ทำไมต้องใช้ระบบหลายตัวแทน?

งานในโลกจริงอย่างการวางแผนการเดินทางเกี่ยวข้องกับความเชี่ยวชาญหลากหลายประเภท — โลจิสติกส์ ความรู้ท้องถิ่น งบประมาณ และอื่นๆ ตัวแทนเดียวที่พยายามจัดการทุกอย่างมักจะยากต่อการควบคุม

ระบบหลายตัวแทนแก้ไขปัญหานี้ด้วย **ความเชี่ยวชาญเฉพาะด้าน**: แต่ละตัวแทนมุ่งเน้นในพื้นที่ความเชี่ยวชาญหนึ่ง ๆ ผลลัพธ์ที่ได้จึงมีคุณภาพสูงกว่าตัวแทนทั่วไป นอกจากนี้ยังช่วยเพิ่ม **ความสามารถในการขยาย** — คุณสามารถเพิ่มตัวแทนใหม่ (เช่น ผู้เชี่ยวชาญด้านเที่ยวบิน นักวิจารณ์ร้านอาหาร) ได้โดยไม่ต้องเขียนโปรแกรมกระบวนงานเดิมใหม่ ตัวแทนเหล่านี้ทำงานร่วมกันผ่านสายการทำงานที่มีโครงสร้าง โดยส่งต่อบริบทจากตัวแทนหนึ่งไปยังอีกตัวแทนหนึ่ง


## การสร้างเอเจนต์เฉพาะทาง


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## การสร้างเวิร์กโฟลว์ตามลำดับ

`WorkflowBuilder` ช่วยให้คุณเชื่อมตัวแทนเข้ากับกราฟที่มีทิศทางได้ ที่นี่เราจะสร้างสายงานสองขั้นตอนอย่างง่าย: **TravelPlanner** ร่างแผนการเดินทาง จากนั้น **TravelConcierge** จะตรวจทานและปรับปรุงมันต่อไป


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## การเพิ่มตัวแทนมากขึ้นในเวิร์กโฟลว์

หนึ่งในข้อได้เปรียบที่ใหญ่ที่สุดของรูปแบบตัวแทนหลายตัวคือความง่ายในการขยาย ด้านล่างนี้เราจะเพิ่มตัวแทน **BudgetReviewer** ที่ตรวจสอบแผนกับงบประมาณของนักเดินทาง ติดธงรายการที่อาจทำให้ค่าใช้จ่ายเกินขีดจำกัด และแนะนำทางเลือกที่ช่วยประหยัดเงิน เวิร์กโฟลว์ตอนนี้รันตัวแทนสามตัวตามลำดับ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## สรุป

ในบทเรียนนี้ คุณได้เรียนรู้วิธี:

1. **สร้างเอเจนต์เฉพาะทาง** — แต่ละตัวมีบทบาทที่เน้นเฉพาะ (การวางแผน, คอนเซียร์จ, การตรวจสอบงบประมาณ)
2. **เชื่อมต่อเอเจนต์ให้เป็นเวิร์กโฟลว์ตามลำดับ** โดยใช้ `WorkflowBuilder` และ `add_edge`
3. **สตรีมเอาต์พุต** จากท่อส่งข้อมูลแบบหลายเอเจนต์ โดยติดตามว่าเอเจนต์ใดกำลังพูดอยู่
4. **ขยายเวิร์กโฟลว์** โดยเพิ่มเอเจนต์ใหม่ในสายงานโดยไม่ต้องแก้ไขเอเจนต์ที่มีอยู่

รูปแบบการออกแบบหลายเอเจนต์ช่วยให้เอเจนต์แต่ละตัวมีความเรียบง่ายในขณะที่ได้ผลลัพธ์ที่สมบูรณ์และผ่านการทบทวนอย่างละเอียดมากกว่าที่เอเจนต์เดียวจะทำได้เพียงลำพัง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด กรุณาทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้องได้ เอกสารต้นฉบับในภาษาต้นทางควรถือเป็นแหล่งข้อมูลที่น่าเชื่อถือสำหรับข้อมูลสำคัญ หากเป็นข้อมูลที่มีความสำคัญ แนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญที่เป็นมนุษย์ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดขึ้นจากการใช้การแปลฉบับนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
